# 03. 에이전트 무한 루프와 비용 폭주 (OWASP LLM06 / ASI08)

| 항목 | 내용 |
|---|---|
| OWASP | **LLM06:2026 Unbounded Consumption** (자원·비용 사용에 상한이 없음) / **ASI08 Cascading Failures** (오류가 연쇄 확대) |
| 원본 위치 | `05LangGraph/03·04·05·06` 의 `create_agent` / `StateGraph` 호출, `get_llm()` 의 `max_retries=5` |
| 위험도 | 중간~높음 (비용 폭주, 서비스 지연, 한도 소진 — 데이터 유출은 아님) |
| LLM 호출 | 없음 — 무한히 도구를 호출하는 **가짜 모델**을 사용하므로 API 키·비용이 필요 없습니다 |

## 핵심 개념
- 에이전트는 `모델 호출 → 도구 실행 → 모델 호출 …` 을 **모델이 "그만"이라고 할 때까지** 반복합니다.
- 도구가 계속 "결과 없음, 다시 시도"를 돌려주거나, 프롬프트 인젝션으로 모델이 계속 도구를 부르면 **끝나지 않습니다**.
- 이때 막아 주는 것은 **개발자가 정한 상한**뿐입니다. 그런데 설치된 LangGraph(1.0.6)의 기본 `recursion_limit` 은 **10000** 이라 사실상 무제한입니다.
- 학습 코드에서 `recursion_limit`, `max_tokens` 를 지정한 곳은 없습니다. (06의 재작성 루프만 직접 만든 횟수 제한이 있음)

## 이 노트북의 흐름
1. Before: 한도 없는 에이전트가 얼마나 호출하는지 측정
2. After ①: `recursion_limit` 로 그래프 단계 상한
3. After ②: `ModelCallLimitMiddleware` 로 모델 호출 횟수 상한 (정상 종료)
4. After ③: 입력 길이 제한 + LLM 파라미터(`max_tokens`, `timeout`, `max_retries`)
5. 전/후 비교 + 자동 검증(`assert`)

In [ ]:
import os
import warnings

import pandas as pd
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langgraph.errors import GraphRecursionError

warnings.filterwarnings("ignore")

# LangGraph 의 기본 단계 상한. 설치된 1.0.6 소스 기준 10000 이고 환경 변수로 바꿀 수 있다 (버전에 따라 다를 수 있음).
# langgraph._internal 은 비공개 모듈이라 import 하지 않고, 같은 규칙으로 값을 읽어 표시만 한다.
DEFAULT_RECURSION_LIMIT = int(os.getenv("LANGGRAPH_DEFAULT_RECURSION_LIMIT", "10000"))

print("LangGraph 기본 recursion_limit:", DEFAULT_RECURSION_LIMIT)

## 0. 이 노트북에서 처음 나오는 클래스와 함수

앞 노트북(01·02)에서 쓰지 않은 것들입니다. **실무에서 쓰는 것**과 **실습용 가짜 모델을 만들기 위한 것**으로 나뉩니다.
(설명은 설치된 버전 langgraph 1.0.6 / langchain 1.2.6 의 소스와 docstring 을 기준입니다.)

| 이름 | 한 줄 역할 | 이 노트북에서 |
|---|---|---|
| `ModelCallLimitMiddleware` | 에이전트의 **모델 호출 횟수에 상한**을 두는 부품 | 방어 수단 (After ②) |
| `GraphRecursionError` | 그래프가 **정해진 단계 수를 넘으면** 발생하는 예외 | 방어 수단 (After ①) |
| `DEFAULT_RECURSION_LIMIT` | LangGraph 의 **기본 단계 상한 값** (10000) | 문제 확인 (Before) |
| `BaseChatModel` | 모든 챗 모델의 **부모 클래스** | 가짜 모델의 뼈대 |
| `ChatResult` | 모델 한 번의 응답을 담는 **상자** | 가짜 모델의 반환값 |
| `ChatGeneration` | `ChatResult` 안에 들어가는 **응답 한 건** | 가짜 모델의 반환값 |

### 실무에서 쓰는 것

**`ModelCallLimitMiddleware`** (`langchain.agents.middleware`)
- 에이전트가 모델을 몇 번까지 부를 수 있는지 제한하고, 넘으면 에이전트를 멈춥니다.
- **미들웨어(middleware)** 는 `모델 호출 → 도구 실행` 루프 중간에 끼어들어 개입하는 부품입니다. `create_agent(..., middleware=[...])` 로 붙입니다.
- 주요 인자: `run_limit`(한 번의 `invoke` 당 상한), `thread_limit`(같은 대화 전체의 누적 상한), `exit_behavior`(`"end"`: 예외 없이 안내 메시지를 남기고 정상 종료 / `"error"`: 예외 발생)
- 이 노트북에서는 `run_limit=5, exit_behavior="end"` 로 5회에서 멈추고 `Model call limits exceeded: run limit (5/5)` 메시지를 남깁니다.
- `thread_limit` 은 docstring 상 "여러 번 invoke 해도 호출 수가 누적"되는 방식입니다.

**`GraphRecursionError`** (`langgraph.errors`)
- 그래프가 한 번의 실행에서 거칠 수 있는 **단계(step) 수**를 넘으면 발생하는 예외입니다. 무한 루프를 막는 안전장치입니다.
- 파이썬의 재귀 호출과는 **상관없습니다.** 여기서 recursion 은 "그래프 단계 반복"을 뜻합니다. (부모 클래스가 `RecursionError`, `RuntimeError` 일 뿐입니다.)
- `config={"recursion_limit": N}` 의 N 을 넘으면 발생합니다. `try/except` 로 잡아서 사용자에게 보여줄 안내 문구로 바꿉니다. (3장의 `run_with_step_limit()`)

**`DEFAULT_RECURSION_LIMIT`**
- `recursion_limit` 을 지정하지 않았을 때 LangGraph 가 쓰는 기본값입니다. 설치된 1.0.6 에서는 **10000** 이라 사실상 무제한입니다. 그래서 Before 에서 모델이 300번 넘게 호출될 때까지 아무도 막지 못합니다.
- 환경 변수 `LANGGRAPH_DEFAULT_RECURSION_LIMIT` 으로 바꿀 수 있습니다.
- LangGraph 안에서는 `langgraph._internal` (**비공개 모듈**)에 정의되어 있어서, 이 노트북은 직접 import 하지 않고 같은 규칙으로 값을 읽어 표시만 합니다. 실제 동작은 Before 의 호출 횟수로 확인합니다.

### 실습용 가짜 모델에 쓴 것

LLM 호출 없이 "끝나지 않는 에이전트"를 재현하려고, 항상 도구만 호출하는 가짜 모델(`LoopingFakeModel`)을 직접 만듭니다.

```
LoopingFakeModel(BaseChatModel)            ← 챗 모델의 부모 클래스를 상속
  └ _generate() 가 ChatResult 를 반환
        └ ChatResult(generations=[ChatGeneration(message=AIMessage(...))])
```

| 클래스 | 설명 |
|---|---|
| `BaseChatModel` | `ChatOpenAI`, `ChatAnthropic` 등의 공통 부모입니다. `invoke`, `stream`, `bind_tools` 같은 공통 기능이 들어 있습니다. 상속해서 `_generate()`(실제 응답을 만드는 메서드)와 `_llm_type`(모델 종류 이름)만 구현하면 `create_agent` 가 진짜 모델처럼 다룹니다. |
| `ChatResult` | 모델 호출 한 번의 결과를 담는 컨테이너이며 `generations` 목록을 가집니다. |
| `ChatGeneration` | `ChatResult.generations` 에 들어가는 항목이며, 실제 내용은 `message`(대개 `AIMessage`)에 담깁니다. |

`invoke()` 를 부르면 LangChain 이 `ChatResult` 를 열어 안의 `AIMessage` 를 꺼내 돌려줍니다. 그래서 평소에는 `ChatResult`, `ChatGeneration` 을 볼 일이 없고, **모델을 직접 구현할 때만** 이 두 클래스로 응답을 포장합니다. 아래 `bind_tools()` 가 `return self` 인 것은 "도구를 바인딩하는 척만 하고 항상 같은 도구를 호출"하려는 의도입니다.

## 1. 실습용 가짜 모델과 도구

- `LoopingFakeModel`: **항상 도구를 호출하겠다고만 답하는** 모델입니다. (끝내지 않는 에이전트를 모의)
- `demo_cutoff`: 실습 노트북이 멈추지 않도록 둔 **데모 안전장치**입니다. 실제 서비스에는 이런 장치가 없다는 점이 문제입니다.
- 호출 1회당 토큰 수(`TOKENS_PER_CALL`)와 단가(`PRICE_PER_1M_TOKENS`)는 **설명을 위한 가정값**입니다. 실제 요금이 아닙니다.

In [ ]:
TOKENS_PER_CALL = 2000          # 가정: 호출 1회당 입력+출력 토큰 (대화가 길수록 실제로는 더 커짐)
PRICE_PER_1M_TOKENS = 1.0       # 가정: 100만 토큰당 1달러 (예시 단가)


class LoopingFakeModel(BaseChatModel):
    """항상 search_docs 도구를 호출하려는 모델 (종료하지 않는 에이전트를 재현)"""
    calls: int = 0
    demo_cutoff: int = 300      # 데모 안전장치: 이 횟수를 넘으면 강제로 예외 발생

    @property
    def _llm_type(self) -> str:
        return "looping-fake"

    def bind_tools(self, tools, **kwargs):
        return self             # 도구 바인딩은 무시 (항상 같은 도구만 호출)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self.calls += 1
        if self.calls > self.demo_cutoff:
            raise RuntimeError(f"데모 안전장치 작동: {self.demo_cutoff}회 호출 후 강제 중단")
        msg = AIMessage(
            content="",
            tool_calls=[{"name": "search_docs", "args": {"query": f"재시도 {self.calls}"}, "id": f"call_{self.calls}"}],
        )
        return ChatResult(generations=[ChatGeneration(message=msg)])


@tool
def search_docs(query: str) -> str:
    """문서를 검색한다. (실습용: 항상 결과가 없다고 답해 에이전트가 계속 재시도하게 만든다)"""
    return "검색 결과가 없습니다. 다른 검색어로 다시 시도하세요."


def estimate_cost(calls: int) -> str:
    """호출 횟수로 토큰과 비용을 어림 계산 (가정값 기반)"""
    tokens = calls * TOKENS_PER_CALL
    return f"{tokens:,} 토큰 / 약 ${tokens / 1_000_000 * PRICE_PER_1M_TOKENS:.2f}"


QUESTION = {"messages": [("user", "문서에서 환불 규정을 찾아 줘")]}

## 2. Before — 한도 없는 에이전트 (원본과 같은 구성)

`create_agent(model, tools)` 만 만들고 `invoke` 합니다. `recursion_limit` 을 지정하지 않았으므로 기본값(10000)이 적용됩니다.

In [ ]:
model_before = LoopingFakeModel(demo_cutoff=300)
agent_before = create_agent(model=model_before, tools=[search_docs])

try:
    agent_before.invoke(QUESTION)           # 한도 지정 없음
    print("정상 종료")
except RuntimeError as e:
    print("중단 원인:", e)

print(f"모델 호출 횟수: {model_before.calls}회  (그래프 한도 {DEFAULT_RECURSION_LIMIT} 은 작동하지 않음)")
print("예상 사용량:", estimate_cost(min(model_before.calls, model_before.demo_cutoff)))

> **결과 해석**: 질문 1개에 모델이 300번 이상 호출될 때까지 아무도 막지 못했습니다. 데모 안전장치가 없었다면 계속 이어집니다.
> 호출이 쌓일수록 대화 기록이 길어져서 **호출당 토큰도 계속 늘어납니다**. (위 비용은 오히려 적게 잡은 값)
> 요청이 동시에 여러 개 들어오면 같은 문제가 곱절로 커집니다.

## 3. After ① — `recursion_limit` 으로 그래프 단계 상한

- `config={"recursion_limit": N}` 은 한 번의 실행에서 **거칠 수 있는 단계(step) 수**를 제한합니다. 넘으면 `GraphRecursionError` 가 발생합니다.
- `create_agent` 에서는 모델 호출 1회와 도구 실행 1회가 각각 1단계라서, 모델 호출은 대략 N/2 회까지입니다.
- 에러를 **그냥 터뜨리지 말고** 사용자에게 보여줄 메시지로 바꿔 줍니다.

In [ ]:
MAX_GRAPH_STEPS = 10     # 상한: 업무에 필요한 최대 단계 수를 측정해서 정할 것 (여유 있게, 하지만 작게)


def run_with_step_limit(agent, payload: dict, limit: int = MAX_GRAPH_STEPS) -> str:
    """단계 상한을 적용해 실행하고, 초과 시 안내 문구를 돌려준다."""
    try:
        result = agent.invoke(payload, config={"recursion_limit": limit})
        return result["messages"][-1].content
    except GraphRecursionError:
        return "처리 단계가 너무 많아 중단했습니다. 질문을 더 구체적으로 다시 입력해 주세요."


model_limit = LoopingFakeModel(demo_cutoff=300)
agent_limit = create_agent(model=model_limit, tools=[search_docs])

answer_limit = run_with_step_limit(agent_limit, QUESTION)
print("사용자에게 보이는 응답:", answer_limit)
print(f"모델 호출 횟수: {model_limit.calls}회 / 예상 사용량: {estimate_cost(model_limit.calls)}")

## 4. After ② — `ModelCallLimitMiddleware` 로 모델 호출 횟수 상한

- LangChain 1.x 의 에이전트 미들웨어(middleware)입니다. **모델 호출 횟수**를 직접 제한하므로 단계 수로 환산할 필요가 없습니다.
- `run_limit`: 한 번의 실행(invoke)당 상한 / `thread_limit`: 같은 대화(thread) 전체 상한 (체크포인터 사용 시)
- `exit_behavior="end"`: 예외 없이 **정상 종료하며 안내 메시지**를 남깁니다. (`"error"` 는 예외 발생)
- 단계 상한(`recursion_limit`)은 **마지막 안전망**으로 함께 두는 것이 좋습니다.

In [ ]:
MAX_MODEL_CALLS = 5

model_mw = LoopingFakeModel(demo_cutoff=300)
agent_mw = create_agent(
    model=model_mw,
    tools=[search_docs],
    middleware=[ModelCallLimitMiddleware(run_limit=MAX_MODEL_CALLS, exit_behavior="end")],
)

result_mw = agent_mw.invoke(QUESTION, config={"recursion_limit": 25})   # 안전망 병행
print("마지막 메시지:", result_mw["messages"][-1].content)
print(f"모델 호출 횟수: {model_mw.calls}회 / 예상 사용량: {estimate_cost(model_mw.calls)}")

## 5. After ③ — 입력 길이 제한과 LLM 파라미터

루프가 아니어도 **입력이 아주 길거나, 출력이 끝없이 길어지거나, 응답이 오지 않거나, 재시도가 쌓이면** 비용·지연이 커집니다.

| 항목 | 위험 | 대책 |
|---|---|---|
| 입력 길이 | 거대한 입력으로 토큰 소모 | 문자 수/토큰 수 상한 후 거부 |
| `max_tokens` | 출력이 끝없이 길어짐 | 업무에 맞는 상한 지정 |
| `timeout` | 응답 지연 시 요청이 계속 대기 | 초 단위 상한 |
| `max_retries` | 장애 때 호출이 N배로 늘어남 (원본은 5) | 2 안팎으로 축소, 지수 백오프 |

In [ ]:
MAX_INPUT_CHARS = 2000      # 질문 길이 상한 (서비스 성격에 맞게 조정)


def validate_user_input(text: str, max_chars: int = MAX_INPUT_CHARS) -> str:
    """사용자 입력을 검증한다. 비어 있거나 너무 길면 ValueError."""
    text = (text or "").strip()
    if not text:
        raise ValueError("질문이 비어 있습니다")
    if len(text) > max_chars:
        raise ValueError(f"질문이 너무 깁니다 ({len(text):,}자 > {max_chars:,}자)")
    return text


for sample in ["환불 규정이 뭐예요?", "가" * 100_000, "   "]:
    try:
        validate_user_input(sample)
        print(f"통과: {sample[:20]!r}")
    except ValueError as e:
        print(f"거부: {e}")

In [ ]:
from langchain_openai import ChatOpenAI

# 네트워크 호출 없이 "설정값"만 확인하기 위한 객체 (api_key 는 실습용 자리표시자이며 실제 호출하지 않음)
llm_safe = ChatOpenAI(
    model="gpt-4o-mini",
    api_key="offline-demo-placeholder",
    max_tokens=1024,     # 출력 토큰 상한
    timeout=30,          # 요청 제한 시간(초)
    max_retries=2,       # 원본의 5회 → 2회
)
print("max_tokens :", llm_safe.max_tokens)
print("timeout    :", llm_safe.request_timeout)
print("max_retries:", llm_safe.max_retries)

# 원본 get_llm() 에 적용하는 방법 예시 (kwargs 로 그대로 전달됨):
#   llm = get_llm(temperature=0, max_tokens=1024, timeout=30)
# MAX_RETRIES 상수도 5 → 2 로 낮추는 것을 권장

## 6. 전/후 비교 및 자동 검증

In [ ]:
rows = [
    {"구성": "Before (한도 없음)", "모델 호출": model_before.calls, "종료 방식": "데모 안전장치로 강제 중단",
     "예상 사용량": estimate_cost(model_before.calls)},
    {"구성": "After ① recursion_limit=10", "모델 호출": model_limit.calls, "종료 방식": "GraphRecursionError → 안내 문구",
     "예상 사용량": estimate_cost(model_limit.calls)},
    {"구성": "After ② ModelCallLimit(5)", "모델 호출": model_mw.calls, "종료 방식": "정상 종료 + 안내 메시지",
     "예상 사용량": estimate_cost(model_mw.calls)},
]
pd.DataFrame(rows)

In [ ]:
# 자동 검증
assert model_before.calls > 100, "Before 에서 과다 호출이 재현되지 않았습니다"
assert model_limit.calls <= MAX_GRAPH_STEPS // 2 + 1, "recursion_limit 이 호출 수를 제한하지 못했습니다"
assert "중단" in answer_limit
assert model_mw.calls <= MAX_MODEL_CALLS, "미들웨어 상한을 넘었습니다"
assert llm_safe.max_tokens == 1024 and llm_safe.max_retries == 2

reduction = 1 - model_mw.calls / model_before.calls
print(f"검증 통과: 모델 호출 {model_before.calls}회 → {model_mw.calls}회 ({reduction:.0%} 감소)")

## 7. 바뀐 점 요약

| 구분 | Before | After |
|---|---|---|
| 그래프 단계 상한 | 기본값 10000 (사실상 없음) | `recursion_limit=10` + 오류를 안내 문구로 변환 |
| 모델 호출 상한 | 없음 | `ModelCallLimitMiddleware(run_limit=5)` |
| 입력 길이 | 제한 없음 | 2000자 상한, 빈 입력 거부 |
| 출력 길이 | 모델 기본값 | `max_tokens=1024` |
| 응답 대기 / 재시도 | 기본값 / `max_retries=5` | `timeout=30` / `max_retries=2` |

## 8. 남는 위험과 추가 대책
- **상한 값은 측정해서 정합니다.** 너무 작으면 정상 질문도 끊깁니다. 정상 요청이 몇 단계를 쓰는지 먼저 확인하세요.
- **사용자별 호출 횟수 제한(rate limit)** 과 **일일 예산 한도**가 별도로 필요합니다. 공개 챗봇(Gradio 등)은 요청 자체가 반복될 수 있습니다.
- **제공자 대시보드의 월 예산·알림 설정**이 마지막 안전장치입니다. 코드가 뚫려도 청구 상한은 막아 줍니다.
- 같은 질문에 대한 **캐시**, 요청 **동시 실행 수 제한**, 도구 실행 **타임아웃**도 함께 검토하세요.
- 상한에 걸린 요청은 **로그**로 남겨 두면 공격이나 오작동을 발견하는 단서가 됩니다.